# Project Deliverable

This notebook corresponds to the development of the project deliverable for the course "Data Analytics - GRAD 50800". For this project I choose to use **"Home Mortgage Disclosure Act (HMDA) Loan/Application Register Data"**

First we will do a quick review of the data:

In [6]:
import polars as pl

PATH = "project_deliverable/data/year_2025.csv"

# 1. Lazy scan: nothing is loaded yet.
#    infer_schema=False reads every column as text, which avoids type errors
#    from values like "Exempt" and "NA" mixed in with numbers.

lf = pl.scan_csv(PATH, infer_schema=False)

# 2. Quick look at the first rows of the nationwide file (reads only 5 rows)
print(lf.head(5).collect())

shape: (5, 99)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ activity_ ┆ lei       ┆ derived_m ┆ state_cod ┆ … ┆ tract_to_ ┆ tract_own ┆ tract_one ┆ tract_me │
│ year      ┆ ---       ┆ sa-md     ┆ e         ┆   ┆ msa_incom ┆ er_occupi ┆ _to_four_ ┆ dian_age │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ e_percent ┆ ed_units  ┆ family_ho ┆ _of_hous │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ age       ┆ ---       ┆ mes       ┆ ing_un…  │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025      ┆ 7H6GLXDRU ┆ 40140     ┆ CA        ┆ … ┆ 86        ┆ 861       ┆ 1173      ┆ 63       │
│           ┆ GQFU57RNE ┆           ┆           ┆   ┆           ┆           

In [7]:
# Clean header names: strip spaces, a possible BOM, and lowercase
names = lf.collect_schema().names()
clean = {n: n.strip().lstrip("\ufeff").lower() for n in names}
lf = lf.rename(clean)
available = set(clean.values())

In [8]:
print(lf.head(5).collect())

shape: (5, 99)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ activity_ ┆ lei       ┆ derived_m ┆ state_cod ┆ … ┆ tract_to_ ┆ tract_own ┆ tract_one ┆ tract_me │
│ year      ┆ ---       ┆ sa-md     ┆ e         ┆   ┆ msa_incom ┆ er_occupi ┆ _to_four_ ┆ dian_age │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ e_percent ┆ ed_units  ┆ family_ho ┆ _of_hous │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ age       ┆ ---       ┆ mes       ┆ ing_un…  │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025      ┆ 7H6GLXDRU ┆ 40140     ┆ CA        ┆ … ┆ 86        ┆ 861       ┆ 1173      ┆ 63       │
│           ┆ GQFU57RNE ┆           ┆           ┆   ┆           ┆           

In [9]:
available

{'action_taken',
 'activity_year',
 'applicant_age',
 'applicant_age_above_62',
 'applicant_credit_score_type',
 'applicant_ethnicity-1',
 'applicant_ethnicity-2',
 'applicant_ethnicity-3',
 'applicant_ethnicity-4',
 'applicant_ethnicity-5',
 'applicant_ethnicity_observed',
 'applicant_race-1',
 'applicant_race-2',
 'applicant_race-3',
 'applicant_race-4',
 'applicant_race-5',
 'applicant_race_observed',
 'applicant_sex',
 'applicant_sex_observed',
 'aus-1',
 'aus-2',
 'aus-3',
 'aus-4',
 'aus-5',
 'balloon_payment',
 'business_or_commercial_purpose',
 'census_tract',
 'co-applicant_age',
 'co-applicant_age_above_62',
 'co-applicant_credit_score_type',
 'co-applicant_ethnicity-1',
 'co-applicant_ethnicity-2',
 'co-applicant_ethnicity-3',
 'co-applicant_ethnicity-4',
 'co-applicant_ethnicity-5',
 'co-applicant_ethnicity_observed',
 'co-applicant_race-1',
 'co-applicant_race-2',
 'co-applicant_race-3',
 'co-applicant_race-4',
 'co-applicant_race-5',
 'co-applicant_race_observed',
 'co-ap

In [10]:
# 3. Keep Tippecanoe County, IN, plus a set of useful columns
cols = ["activity_year", "state_code", "county_code", "action_taken", "loan_type",
        "loan_purpose", "loan_amount", "property_value", "income",
        "debt_to_income_ratio", "loan_to_value_ratio",
        "interest_rate", "loan_term", "derived_race", "derived_sex", "applicant_age"]

county = (
    lf.filter(pl.col("county_code") == "18157")
      .select(cols)
      .with_columns(
          pl.col(["loan_amount", "property_value", "income",
                  "loan_to_value_ratio", "interest_rate", "loan_term"])
            .cast(pl.Float64, strict=False)   # "NA"/"Exempt" become null
      )
)

In [12]:
# 4. Run the query with the streaming engine (processes the file in chunks)
df = county.collect(engine="streaming")
print(df.shape)
print(df.head(10))

(7124, 16)
shape: (10, 16)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ activity_ ┆ state_cod ┆ county_co ┆ action_ta ┆ … ┆ loan_term ┆ derived_r ┆ derived_s ┆ applican │
│ year      ┆ e         ┆ de        ┆ ken       ┆   ┆ ---       ┆ ace       ┆ ex        ┆ t_age    │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ f64       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025      ┆ IN        ┆ 18157     ┆ 6         ┆ … ┆ 654.0     ┆ Race Not  ┆ Sex Not   ┆ 8888     │
│           ┆           ┆           ┆           ┆   ┆           ┆ Available ┆ Available ┆          │
│ 2025      ┆ IN        ┆ 18157     ┆ 6         ┆ … ┆ 240.0     ┆ Race Not  ┆ Sex Not   ┆ 8888     │
│           ┆           ┆           ┆           ┆   ┆           